In [ ]:
# Load or reload R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Complete Engineered Feature Distribution Visualizations (`models/plot_engineered.ipynb`)

This notebook constructs and plots the distributions of **ALL 26 FEATURE-ENGINEERED INPUTS** across the 5 Emergency Severity Index (ESI) Triage Classes (`1`, `2`, `3`, `4`, `5`):

### 10 Clinical Binary Indicator Features (Individual Bar Plots):
1. `is_dyspnea_total`: `triage_vital_o2 < 90`
2. `is_dyspnea_moderate`: `90 < triage_vital_o2 < 94`
3. `is_bradypnea`: `triage_vital_rr < 10`
4. `is_tachypnea`: `triage_vital_rr > 30`
5. `is_hypotension`: `triage_vital_sbp <= 90`
6. `is_hypertension`: `triage_vital_sbp > 220`
7. `is_bradycardia_total`: `triage_vital_hr < 40`
8. `is_bradycardia_moderate`: `40 < triage_vital_hr < 60`
9. `is_tachycardia_total`: `triage_vital_hr > 150`
10. `is_tachycardia_moderate`: `100 < triage_vital_hr < 150`

### 16 Vital Delta & Range Features (Individual Boxplots):
11. `hr_mean_to_last`: `triage_vital_hr - pulse_last`
12. `sbp_mean_to_last`: `triage_vital_sbp - sbp_last`
13. `spo2_mean_to_last`: `triage_vital_o2 - spo2_last`
14. `rr_mean_to_last`: `triage_vital_rr - resp_last`
15. `hr_range`: `pulse_max - pulse_min`
16. `rr_range`: `resp_max - resp_min`
17. `spo2_range`: `spo2_max - spo2_min`
18. `sbp_range`: `sbp_max - sbp_min`
19. `hr_last_to_min`: `pulse_last - pulse_min`
20. `sbp_last_to_min`: `sbp_last - sbp_min`
21. `spo2_last_to_min`: `spo2_last - spo2_min`
22. `rr_last_to_min`: `resp_last - resp_min`
23. `hr_last_to_max`: `pulse_last - pulse_max`
24. `sbp_last_to_max`: `sbp_last - sbp_max`
25. `spo2_last_to_max`: `spo2_last - spo2_max`
26. `rr_last_to_max`: `resp_last - resp_max`

All 26 plots are saved individually into `plots/engineered_feature_distributions/`.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Data & Construct ALL 26 Engineered Features in R
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(dplyr)
  library(ggplot2)
  library(tidyr)
})
config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) config_path <- "config/triage_conf.json"
config <- fromJSON(config_path)
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) data_file <- paste0("../", data_file)
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
raw_df   <- get(df_names[which.max(df_sizes)], envir = data_env)
target_col_name <- config$classes$target_col
get_vec <- function(col_name, default_val = 0) {
  if (col_name %in% names(raw_df)) {
    res <- raw_df[[col_name]]
    res[is.na(res)] <- default_val
    return(res)
  } else {
    return(rep(default_val, nrow(raw_df)))
  }
}
pulse_last <- get_vec("pulse_last"); pulse_max <- get_vec("pulse_max"); pulse_min <- get_vec("pulse_min")
sbp_last   <- get_vec("sbp_last");   sbp_max   <- get_vec("sbp_max");   sbp_min   <- get_vec("sbp_min")
spo2_last  <- get_vec("spo2_last");  spo2_max  <- get_vec("spo2_max");  spo2_min  <- get_vec("spo2_min")
resp_last  <- get_vec("resp_last");  resp_max  <- get_vec("resp_max");  resp_min  <- get_vec("resp_min")
t_hr       <- get_vec("triage_vital_hr"); t_sbp <- get_vec("triage_vital_sbp"); t_o2 <- get_vec("triage_vital_o2"); t_rr <- get_vec("triage_vital_rr")
df_eng <- data.frame(
  esi                    = factor(as.character(raw_df[[target_col_name]]), levels = c("1", "2", "3", "4", "5")),
  # 10 Clinical Binary Indicators
  is_dyspnea_total       = ifelse(t_o2 < 90, 1, 0),
  is_dyspnea_moderate    = ifelse(t_o2 > 90 & t_o2 < 94, 1, 0),
  is_bradypnea           = ifelse(t_rr < 10, 1, 0),
  is_tachypnea           = ifelse(t_rr > 30, 1, 0),
  is_hypotension         = ifelse(t_sbp <= 90, 1, 0),
  is_hypertension         = ifelse(t_sbp > 220, 1, 0),
  is_bradycardia_total   = ifelse(t_hr < 40, 1, 0),
  is_bradycardia_moderate= ifelse(t_hr > 40 & t_hr < 60, 1, 0),
  is_tachycardia_total   = ifelse(t_hr > 150, 1, 0),
  is_tachycardia_moderate= ifelse(t_hr > 100 & t_hr < 150, 1, 0),
  # 16 Vital Delta & Range Features
  hr_mean_to_last        = t_hr - pulse_last,
  sbp_mean_to_last       = t_sbp - sbp_last,
  spo2_mean_to_last      = t_o2 - spo2_last,
  rr_mean_to_last        = t_rr - resp_last,
  hr_range               = pulse_max - pulse_min,
  rr_range               = resp_max - resp_min,
  spo2_range             = spo2_max - spo2_min,
  sbp_range              = sbp_max - sbp_min,
  hr_last_to_min         = pulse_last - pulse_min,
  sbp_last_to_min        = sbp_last - sbp_min,
  spo2_last_to_min       = spo2_last - spo2_min,
  rr_last_to_min         = resp_last - resp_min,
  hr_last_to_max         = pulse_last - pulse_max,
  sbp_last_to_max        = sbp_last - sbp_max,
  spo2_last_to_max       = spo2_last - spo2_max,
  rr_last_to_max         = resp_last - resp_max
)
df_eng <- na.omit(df_eng)
df_eng_py <<- df_eng
cat(sprintf("Constructed ALL 26 Engineered Features across %d samples\n", nrow(df_eng)))

In [ ]:
# ---------------------------------------------------------
# Step 2: Generate Individual PNG Plots for ALL 26 Features
# ---------------------------------------------------------
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from rpy2.robjects import r
import rpy2.robjects.pandas2ri as pandas2ri
try:
    pandas2ri.activate()
    df_eng = pd.DataFrame(pandas2ri.rpy2py_dataframe(r['df_eng_py']))
except Exception:
    df_eng = pd.DataFrame(r['df_eng_py'])
out_dir = "../plots/engineered_feature_distributions"
if not os.path.exists(out_dir): out_dir = "plots/engineered_feature_distributions"
os.makedirs(out_dir, exist_ok=True)
binary_feats = [
    'is_dyspnea_total', 'is_dyspnea_moderate', 'is_bradypnea', 'is_tachypnea',
    'is_hypotension', 'is_hypertension', 'is_bradycardia_total', 'is_bradycardia_moderate',
    'is_tachycardia_total', 'is_tachycardia_moderate'
]
continuous_feats = [
    'hr_mean_to_last', 'sbp_mean_to_last', 'spo2_mean_to_last', 'rr_mean_to_last',
    'hr_range', 'rr_range', 'spo2_range', 'sbp_range',
    'hr_last_to_min', 'sbp_last_to_min', 'spo2_last_to_min', 'rr_last_to_min',
    'hr_last_to_max', 'sbp_last_to_max', 'spo2_last_to_max', 'rr_last_to_max'
]
esi_colors = {'1': '#d62728', '2': '#ff7f0e', '3': '#1f77b4', '4': '#2ca02c', '5': '#9467bd'}
print(f"Plotting all 26 engineered features to: {out_dir} ...")
# 1. Plot individual bar plots for 10 binary features
for idx, feat in enumerate(binary_feats, start=1):
    plt.figure(figsize=(7, 4.5), dpi=200)
    prop_df = df_eng.groupby('esi')[feat].mean().reset_index()
    prop_df['Percentage'] = prop_df[feat] * 100
    
    ax = sns.barplot(data=prop_df, x='esi', y='Percentage', palette=esi_colors)
    plt.title(f'Feature [{idx}/26]: {feat} (% Positive per ESI)', fontsize=12, fontweight='bold', pad=10)
    plt.xlabel('ESI Triage Level', fontsize=10)
    plt.ylabel('Percentage Positive (%)', fontsize=10)
    plt.ylim(0, max(prop_df['Percentage'].max() * 1.25, 5))
    plt.grid(True, linestyle=':', alpha=0.5, axis='y')
    
    for p in ax.patches:
        height = p.get_height()
        ax.annotate(f'{height:.2f}%', (p.get_x() + p.get_width() / 2., height),
                    ha='center', va='bottom', fontsize=9, xytext=(0, 3), textcoords='offset points')
        
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{idx:02d}_{feat}.png"))
    plt.close()
# 2. Plot individual boxplots for 16 continuous features
for idx, feat in enumerate(continuous_feats, start=11):
    plt.figure(figsize=(7, 4.5), dpi=200)
    sns.boxplot(data=df_eng, x='esi', y=feat, palette=esi_colors, showfliers=False)
    plt.title(f'Feature [{idx}/26]: {feat} Distribution across ESI', fontsize=12, fontweight='bold', pad=10)
    plt.xlabel('ESI Triage Level', fontsize=10)
    plt.ylabel('Feature Value', fontsize=10)
    plt.grid(True, linestyle=':', alpha=0.5, axis='y')
    
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{idx:02d}_{feat}.png"))
    plt.close()
print(f"SUCCESS: Generated 26 individual distribution PNG plots in {out_dir}!")